# DQL: HAVING

`HAVING` filters grouped results after aggregation has already happened.

Use `HAVING` when the condition depends on a summary value, such as a count, average, minimum, maximum, or total. It is commonly used after `GROUP BY` to keep only the groups that meet a business rule.


## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## HAVING

`HAVING` filters grouped results. In the DataFrame API, aggregate first and then filter the summary rows.

In plain English: `HAVING` is like `WHERE`, but it filters after grouping has already happened.

Use this memory trick:

* `WHERE` filters individual rows before aggregation.
* `HAVING` filters grouped summary rows after aggregation.

Common uses:

* Keep departments with more than a certain number of employees.
* Keep jobs where the average salary is above a target.
* Keep customers whose total purchases exceed a threshold.
* Remove low-volume groups from summary reports.

In this example, Spark first totals salary by department and then keeps only departments whose total salary is high enough.

Watch out: `HAVING` is not a replacement for `WHERE`. If you can filter detail rows before grouping, use `WHERE` first because it reduces the data that must be aggregated.


In [ ]:
dept_summary = emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.sum("sal").alias("total_salary")
)

dept_summary.where(F.col("total_salary") > 15000).orderBy("total_salary", ascending=False).show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
HAVING SUM(sal) > 15000
ORDER BY total_salary DESC
""").show()
